In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install torch transformers datasets peft bitsandbytes py_vncorenlp \
    accelerate trl beir \
    numpy pandas scikit-learn sentencepiece tokenizers tqdm underthesea

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 148.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 32.2 MB/s eta 0:00:00
  Created wheel for py_vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4304 sha256=f59342ffd8b1523c73fcb2797fb34b956433c9887e84a70f8921d0a9b019e916
  Stored in directory: /root/.cache/pip/wheels/db/e5/ff/f4a1b4ece36e8582db1ca71150a34e987e65df50c35974e9bb
Successfully built py_vncorenlp


In [3]:
!git clone https://github.com/Tommachilez/improving-learned-index.git
%cd /content/improving-learned-index

Cloning into 'improving-learned-index'...
remote: Enumerating objects: 1247, done.
remote: Counting objects: 100% (343/343), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 1247 (delta 300), reused 285 (delta 264), pack-reused 904 (from 1)
Receiving objects: 100% (1247/1247), 206.21 KiB | 18.75 MiB/s, done.
Resolving deltas: 100% (912/912), done.
/content/improving-learned-index


In [4]:
dataset_path = "/content/drive/MyDrive/KLTN_Project/Datasets"
doc_mapping = f"{dataset_path}/unique_doc_mapping.csv"
query_mapping = f"{dataset_path}/train_query_mapping.csv"
pretokenized_docs = f"{dataset_path}/vn_mining/corpus_pretokenized.jsonl"
pretokenized_queries = f"{dataset_path}/vn_mining/queries_pretokenized.jsonl"
output_queries = f"{dataset_path}/deeperimpact/train_queries.tsv"
output_docs = f"{dataset_path}/deeperimpact/gemini_expanded_documents.tsv"
output_expansions = f"{dataset_path}/deeperimpact/expansions.csv"
save_dir = "/content/drive/MyDrive/KLTN_Project/Models/vncorenlp_models"

# Merge expansions

In [5]:
!python -m src.deep_impact.scripts.create_training_files \
    --doc_mapping {doc_mapping} \
    --pretokenized_doc {pretokenized_docs} \
    --query_mapping {query_mapping} \
    --pretokenized_queries {pretokenized_queries} \
    --output_queries_tsv {output_queries} \
    --output_docs_tsv {output_docs} \
    --output_expansion_csv {output_expansions}

Converting /content/drive/MyDrive/KLTN_Project/Datasets/train_query_mapping.csv to /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact/train_queries.tsv...
Processing Queries: 129021it [00:02, 55510.98it/s]
Loading raw documents from CSV: /content/drive/MyDrive/KLTN_Project/Datasets/unique_doc_mapping.csv...
Loading pretokenized docs for filtering from: /content/drive/MyDrive/KLTN_Project/Datasets/vn_mining/corpus_pretokenized.jsonl...
Aggregating query terms from /content/drive/MyDrive/KLTN_Project/Datasets/vn_mining/queries_pretokenized.jsonl...
Reading Queries JSONL: 61931it [00:01, 38171.13it/s]
Loading tokenizer: xlm-roberta-base...
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 204kB/s]
config.json: 100% 615/615 [00:00<00:00, 5.73MB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 24.9MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:00<00:00, 44.7MB/s]
Building expanded documents...
- Expanded Docs TSV: /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact/gemi

# Train

In [5]:
scores = f"{dataset_path}/viranker_firstp/scores.pkl.gz"

In [6]:
!torchrun --standalone --nproc_per_node=gpu -m src.deep_impact.train \
  --dataset_path {scores} \
  --queries_path {output_queries} \
  --collection_path {output_docs} \
  --checkpoint_dir /content/drive/MyDrive/KLTN_Project/Models/deeperimpact_xlmr_vifc \
  --max_length 512 \
  --xlmr \
  --batch_size 4 \
  --save_every 5000 \
  --lr 1e-6 \
  --gradient_accumulation_steps 8 \
  --distil_kl \
  --no_beir_eval \
  --use_wandb \
  --vncorenlp_path {save_dir} \
#   --qrels_path /content/drive/MyDrive/KLTN_Project/Datasets/MS_MARCO/qrels.train \


2025-12-27 23:28:01.075009: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-27 23:28:01.092249: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766878081.113508    6878 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766878081.119993    6878 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766878081.136278    6878 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 